# 10 — Reproducibility summary

What this series established, what it could not, and every open data-provenance and implementation caveat — stated in the repository's own five-category classification vocabulary.

**Data mode: REAL DATA.** Every figure below is computed from the genuine Bloomberg CSV exports for this strategy, through production AtlasQuant code. No synthetic, simulated, imputed, or approximated value appears anywhere in this notebook.

Those exports are gitignored, so they are absent in a fresh clone. When they are missing this notebook still executes top to bottom, but every computing cell prints an explicit *real data not present — cannot produce a genuine result* notice instead of a number. It never falls back to synthetic data: a plausible-looking figure derived from invented inputs is not a result (see `CLAUDE.md`).

In [ ]:
import _real_data as rd

print(rd.banner())
AVAILABLE = rd.data_available()


## What this strategy is

Multi-Factor Ranking ML is a **pure cross-sectional ranking system**. Given a quarterly universe it produces one thing: a contiguous 1..N ordering of every instrument that survives validation and the sector exclusion, each with an unbounded LambdaRank margin.

It has **no portfolio**. There are no positions, no weights, no position count, no capital budget, no qualification threshold, no ETF fallback sleeve, no orders and no P&L. None of those concepts exist in the codebase — they were deleted, not disabled. Consequently there is no total return, no Sharpe ratio, no drawdown and no equity curve to report, and the absence of those figures is the design, not a gap.

Its quality is evaluated by **Information Coefficient** — the rank correlation between its ordering and realized forward returns.

## What each notebook established

In [ ]:
established = [
    ("00", "Environment, git commit, config identity, and SHA-256 identity of every raw CSV."),
    ("02", "Real raw data validates: 1,520 tickers, no filed_at before its own quarter_end, "
           "real per-feature missingness reported rather than filled."),
    ("03", "The provider -> Stage 3 normalization boundary runs on the real export; "
           "rejections surface as structured validation issues."),
    ("04", "71-feature observations build per cycle with an asserted point-in-time guarantee; "
           "missing values reach the model as NaN, never imputed. Five raw-level features "
           "are percentile-ranked within their own cycle's cross-section; macro, "
           "categorical, volatility and growth features are deliberately not."),
    ("05", "Graded 0..9 relevance from real realized forward returns, asserted monotone "
           "in mean forward return. Not a binary top-10 label."),
    ("06", "Rolling 3-year window with LambdaRank query groups that partition the matrix "
           "exactly; a real LGBMRanker fits under the eligibility gates."),
    ("07", "Scores are unbounded ranker margins; the decision pipeline yields a "
           "contiguous, gap-free 1..N ranking of the whole surviving cross-section."),
    ("08", "The IC backtest orchestration runs end to end per cycle on real data."),
    ("09", "Headline IC / IC-IR / hit-rate / decile-spread statistics, with no AUC, "
           "Sharpe, return or drawdown field anywhere."),
]
for number, text in established:
    print(f"{number}  {text}")
    print()

## Provenance record for this run

In [ ]:
if AVAILABLE:
    from atlas_quant.strategies.multi_factor_ranking_ml.config import (
        FEATURE_SCHEMA_VERSION,
        STRATEGY_ID,
        STRATEGY_VERSION,
    )

    bundle = rd.slice_bundle()
    print("strategy               ", STRATEGY_ID, STRATEGY_VERSION)
    print("feature schema version ", FEATURE_SCHEMA_VERSION)
    print("config identity        ", rd.config().identity())
    print("git commit             ", rd.git_commit() or "unavailable")
    print("slice                  ", bundle.cycles[0].quarter_start, "..",
          bundle.cycles[-1].quarter_start, f"({len(bundle.cycles)} cycles)")
    print("universe               ", f"{len(bundle.universe):,} instruments")
    print("fundamentals rows      ", f"{bundle.total_fundamentals_rows:,}")
    print("daily price rows       ", f"{bundle.total_price_rows:,}")
    print("load validation issues ", f"{bundle.load_issue_count:,}")
    print("run identity           ", bundle.ic_result.run_identity)
    print()
    for entry in bundle.source_manifest:
        print(f"  {entry.name:<32} {entry.sha256}")
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Open caveats

Stated in the repository's five-category vocabulary (`source_specification_required`, `data_provenance_required`, `implementation_bug`, `data_quality_issue`, `expected_legacy_difference`). The authoritative list lives in `docs/reproducibility_findings.md`; this is the subset these notebooks surfaced directly.

In [ ]:
caveats = [
    ("data_provenance_required",
     "GICS sector is a present-day snapshot applied backwards over each ticker's whole "
     "history. sector_enc and the Materials exclusion are therefore not genuinely "
     "point-in-time. Disclosed, not silently corrected."),
    ("data_provenance_required",
     "The universe is derived from the tickers present in the export as pulled, so it is "
     "survivorship-biased. Every historical statistic in this series inherits that bias, "
     "and it is recorded on each membership record rather than hidden."),
    ("data_provenance_required",
     "available_date is estimated for part of the history rather than observed. Where it "
     "is estimated, the point-in-time cutoff is only as good as that estimate."),
    ("source_specification_required",
     "price_convention is a caller assertion, not something inferrable from "
     "Close_Price.csv. It was confirmed empirically against known splits, but the export "
     "itself does not state it."),
    ("data_quality_issue",
     "Feature coverage is uneven and materially worse in the early history; a handful of "
     "thin-coverage columns were excluded from the 71-feature schema outright."),
    ("expected_legacy_difference",
     "Relevance ties break on descending clipped return then ascending symbol - a "
     "deliberate, reproducible rule. The legacy prototype's tie-break was an accident of "
     "row-insertion order."),
    ("expected_legacy_difference",
     "The binary top-N label, the probability score, the qualification threshold, "
     "position sizing and the ETF fallback sleeve are all deleted. Results are not "
     "expected to match any figure produced under those mechanisms."),
    ("expected_legacy_difference",
     "Five raw-level features (market_cap, volume, net_debt, adjusted_net_debt, "
     "analyst_target_price) are replaced by their within-cycle percentile rank. Adopted "
     "on theory plus directionally consistent but NOT statistically significant evidence "
     "(+0.0028 mean IC, p = 0.273 over 147 cycles) - no improvement is claimed. Results "
     "are not comparable to runs made on raw levels; FEATURE_SCHEMA_VERSION 4 enforces "
     "that. A cross-section with fewer than 2 non-missing values leaves the raw level in "
     "place, which would matter for any future single-name scoring path."),
]
for classification, text in caveats:
    print(f"[{classification}]")
    print(f"   {text}")
    print()

## What this series does NOT establish

- **Not that the strategy works.** The measured window is a handful of recent quarters. A mean IC over that many observations has a very wide confidence interval; no significance test is performed and none would pass at this sample size.
- **Not out-of-sample performance.** Hyperparameters carried over from the previous classifier are untuned, there is no held-out validation split, and the evaluated window is recent and survivorship-biased.
- **Not a tradable result.** IC and the decile spread are descriptive statistics about an ordering. They are gross, cost-free and price-only. Turning a ranking into a portfolio would require position construction, sizing, costs and risk management — none of which this strategy has, by design.
- **Not full-history reproduction.** These notebooks run a bounded recent slice so they stay re-runnable. The full 1980-2026 history is a separate, much longer run.

## Reproducing this

1. Restore the raw Bloomberg exports (see `docs/data_provenance_manifest.md`). Without them every notebook here prints an explicit *cannot produce a genuine result* notice and computes nothing — deliberately, rather than substituting synthetic data.
2. Install the model extra so `lightgbm` is importable.
3. Run notebooks 00 through 10 in order. The first one to need the derived slice builds it once (~3 minutes) and caches it, fingerprinted against the source CSVs' hashes and the strategy config identity; the rest reuse it. Changing any input invalidates the fingerprint and forces a rebuild.
4. `tests/integration/test_multi_factor_ranking_notebook_execution.py` executes all ten against the real data. It is opt-in (`-m production_data`) and skips cleanly when the raw exports are absent.

## Findings

- All ten notebooks execute end to end against real Bloomberg data through production code paths, with no synthetic fixture anywhere in the series.
- Every point-in-time property the strategy depends on is asserted on the real data rather than assumed: no fundamentals row precedes its own quarter end, no feature observation post-dates its cycle's cutoff, and no training label was knowable later than its cycle's cutoff.
- The pure-ranking design is visible throughout: no weight, no capital, no position and no P&L figure appears anywhere, and the serialized backtest result is asserted to contain none.

## Limitations

- The open caveats above are real and unresolved. The survivorship-biased universe and the non-point-in-time sector snapshot in particular bias every historical statistic in this series in an optimistic direction.
- The evaluated sample is a handful of quarters. Nothing here should be read as evidence that the ranking is predictive out of sample.
- These notebooks demonstrate and validate the pipeline. They are not a research result, and they are not an investment recommendation.